# Standing-wave tube: pressure

The 395 Hz measurement contains 16 pressure positions. The active source withholds nine positions and uses seven as observations. Each sampler uses 500 reverse steps. The original computation standardizes the measurement during loading and again at the sampling input, and applies one inverse transform to predictions. These operations are retained from the source notebook.

Run cells in order. Sampling counts below are reverse-diffusion steps; checkpoint numbers identify training state. Data and checkpoints are installed under the repository root.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'tube_pressure'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import os
from sampler.utils import *
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = 'exp_p'
version = 'v21'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


## Measured pressure and normalization


In [ ]:
import numpy as np
import os
from scipy import signal
data_test_path = 'data/tube395_resampled_calibrated.npy'
scalar_path = opt.results_path + '/scalar.npy'
data_raw = np.load(data_test_path)
print(f'Loaded test data shape: {data_raw.shape}')
(mean, std) = np.load(scalar_path, allow_pickle=True)
scalar = lambda x: (x - mean) / std
scalar_inv = lambda x: x * std + mean
x_test = scalar(data_raw)
m = 2
T = opt.num_frames
T_prime = 500
start_t = 0
seed_samples = 1
rnd = np.random.RandomState(seed_samples)
B_block = int(np.ceil((T_prime - m) / (T - m)))
T_prime = B_block * (T - m) + m
print(f'Calculated T_prime: {T_prime}')
num_samples_available = x_test.shape[0]
if start_t + T_prime > x_test.shape[1]:
    print(f'警告: 起始时间 {start_t} + T_prime {T_prime} (共 {start_t + T_prime} 步) 超出了可用时间 {x_test.shape[1]}')
    start_t = x_test.shape[1] - T_prime
    print(f'自动调整起始时间为: {start_t}')
print(f'正在从 T={start_t} 采样到 T={start_t + T_prime}...')
samples = []
for i_batch in range(num_samples_available):
    single_sample_data = x_test[i_batch, start_t:start_t + T_prime]
    samples.append(single_sample_data)
samples = np.stack(samples, axis=0)
s_idx_to_test = 4
if num_samples_available < 5:
    s_idx_to_test = 0
samples = samples[[s_idx_to_test]]
print(f'Final sample shape for generation: {samples.shape}')


## Reconstruction


In [ ]:
outer_loop = 500
denoising_steps = 500
denoising_steps2 = 500
cof = 0.1
seed_sample = 123
obs_type = 'regular'
scale = 4
alpha_case = 0.3
beta = 0.4
gamma = 0.5
physics_params = {'physics_c': 343.0, 'physics_rho0': 1.21, 'physics_dt': 0.0001, 'physics_dx': 0.02, 'reg_coef_wave': cof}
total_sensors = 16
masked_indices_manual = np.array([2, 4, 6, 7, 8, 9, 11, 12, 14])
all_indices = np.arange(total_sensors)
kept_indices = np.setdiff1d(all_indices, masked_indices_manual)
kept_indices = np.sort(kept_indices)
kept_indices_torch = torch.from_numpy(kept_indices).long().to(device)
print(f'Mask Strategy (Manual Design):')
print(f'  Total Sensors Used: {len(kept_indices)}')
print(f'  Kept Indices:   {kept_indices}')
print(f'  Masked Indices: {masked_indices_manual}')
sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
sde1 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps)
sde2 = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=denoising_steps2)
net.eval()
reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
setup_seed(seed_sample)
with torch.no_grad():
    samples_scale = samples.copy()
    samples_scale[:, :, :opt.num_components] = scalar(samples_scale[:, :, :opt.num_components])
    print(f'Num Components: {opt.num_components}')
    y_full = samples_scale
    y = y_full[:, :, :, kept_indices]
    print(f'Observation shape (y): {y.shape}')
    if obs_type.lower() == 'regular':
        T_prime_y = T_prime

        def transform(x):
            downsampled = x[:, :T_prime_y, :opt.num_components, ::scale]
            return downsampled[:, :, :, kept_indices]
        transform_gen = transform
        sparsity = 1 - len(kept_indices) / (16 * scale)
    alpha = alpha_case / np.sqrt(1 - sparsity)
    latent_mask = torch.ones([T, 1, 1]).float().to(device)
    obs_mask = torch.zeros([T, 1, 1]).float().to(device)
    frame_indices = torch.arange(T).to(device)
    net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
    print('>>> Running Method 1: Standard SDE')
    setup_seed(seed_sample)
    x_inpainted = s3gm_sample_1d(opt, net_fn, sde, y, transform_gen, alpha=0.2 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
    print('>>> Running Method 2: ISS (Smoothness)')
    setup_seed(seed_sample)
    x_inpainted1 = s3gm_sample_1d_ISS(opt, net_fn, sde1, y, transform_gen, alpha=0.2 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, denoising_steps=denoising_steps, eta=0, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True, reg_coef_pa=0 * T_prime)
    print('>>> Running Method 3: ISS + Physics Residuals')
    setup_seed(seed_sample)
    x_inpainted2 = s3gm_sample_1d_ISS_1dtube_res_p(opt, net_fn, sde2, y, transform_gen, alpha=0.5 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, denoising_steps=denoising_steps2, eta=0, reg_coef_pa=0 * T_prime, data_scalar=scalar_inv, physics_start_step=denoising_steps2, **physics_params, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
x_pred = x_inpainted
x_pred[:, :, :opt.num_components] = scalar_inv(x_pred[:, :, :opt.num_components])
x_pred1 = x_inpainted1
x_pred1[:, :, :opt.num_components] = scalar_inv(x_pred1[:, :, :opt.num_components])
x_pred2 = x_inpainted2
x_pred2[:, :, :opt.num_components] = scalar_inv(x_pred2[:, :, :opt.num_components])
num_T = x_pred.shape[1]
print(f'Number of generated snapshots: {num_T}')
torch.cuda.empty_cache()


## Held-out pressure measurements


In [ ]:
import numpy as np
masked_indices = masked_indices_manual
scale = 4
pred_indices = masked_indices * scale
print(f'Evaluating error on masked indices: {masked_indices}')
gt_masked = samples[:, :, :, masked_indices]
pred_sde = x_pred[:, :, :, pred_indices]
pred_iss = x_pred1[:, :, :, pred_indices]
pred_pi_iss = x_pred2[:, :, :, pred_indices]
print('-' * 50)
print('Average nRMSE on masked sensors')
print('-' * 50)
err_sde = np.mean(cal_rmse(gt_masked, pred_sde, normalize=True))
print(f'SDE: {err_sde:.6f}')
err_iss = np.mean(cal_rmse(gt_masked, pred_iss, normalize=True))
print(f'ISS: {err_iss:.6f}')
err_pi_iss = np.mean(cal_rmse(gt_masked, pred_pi_iss, normalize=True))
print(f'PI-ISS: {err_pi_iss:.6f}')
print('-' * 50)


## Waveforms and saved metrics


In [ ]:
rows = []
(fig, axes) = plt.subplots(3, 3, figsize=(11, 7), sharex=True)
for (ax, j) in zip(axes.ravel(), masked_indices_manual):
    ax.plot(samples[0, :, 0, j], color='black', label='Measurement')
    for (name, pred) in [('SDE', x_pred), ('ISS', x_pred1), ('PI-ISS', x_pred2)]:
        ax.plot(pred[0, :, 0, j * scale], label=name, alpha=0.8)
    ax.set_title(f'Sensor {j + 1}')
axes.ravel()[0].legend()
format_current_figure()
fig.savefig(OUTPUT / 'pressure_waveforms.pdf', bbox_inches='tight')
format_current_figure()
plt.show()
pd.DataFrame({'method': ['SDE', 'ISS', 'PI-ISS'], 'nrmse': [err_sde, err_iss, err_pi_iss]}).to_csv(OUTPUT / 'pressure_metrics.csv', index=False)
